# Fig. 5 Reproduction — Character-Level Attention Maps

Reproduces a **character-level attention grid** matching Fig. 5 from  
*"Beyond Memorization: Extracting the Trained Knowledge in Deep Neural Networks"*  
(Springer LNCS 14627, ICDAR 2024).

## Key design choices (scientifically motivated)

| Choice | Rationale |
|--------|-----------|
| **5-character word** | Fixed length lets the grid be perfectly uniform without leftover padding |
| **Uniform character crops** | Each character panel is a *fixed-width* slice of the image, making spatial comparisons fair |
| **Gradient saliency** `|∂logit(c,t)/∂x|` | Pixel-level, directly measures each pixel's causal contribution to the CTC peak logit of character *c* |
| **Self-attention (last layer, mean heads)** | Shows *where* the transformer attends when predicting each character |
| **Both methods, single figure** | Enables direct visual comparison of the two interpretability paradigms |

**Layout**: `[Input] | [c₁] | [c₂] | [c₃] | [c₄] | [c₅]`  — repeated for both rows.


In [ ]:
# ── Cell 1 — Imports & paths ─────────────────────────────────────────────────
import sys, json, os
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from scipy.ndimage import zoom as spzoom

# ── Resolve project root ──────────────────────────────────────────────────────
ROOT = Path('.').resolve()
if ROOT.name == 'notebook':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from models import HTRNet
from utils.preprocessing import load_image, preprocess

DEVICE = 'cpu'
IMG_H, IMG_W = 128, 1024
EXP  = ROOT / 'saved_models' / 'experiments'
OUT  = ROOT / 'outputs' / 'attention_maps'
OUT.mkdir(parents=True, exist_ok=True)

charset = list(np.load(str(EXP / 'classes.npy'), allow_pickle=True))
print(f'Charset: {len(charset)} classes | ROOT: {ROOT}')

In [ ]:
# ── Cell 2 — Model & image ────────────────────────────────────────────────────
#
# Using a01-038-12.png ("talks.") — 6 characters, nearest available to 5.
# Set TARGET_LEN = 6 to match the actual decoded output.

RUN        = 'run_63'          # 8 register tokens, CER 6.26 %
IMG_FILE   = 'a01-038-12.png'  # GT: "talks." — 6 chars
TARGET_LEN = 6                 # expected decoded length (non-space chars)

# ── Load model ────────────────────────────────────────────────────────────────
cfg   = json.load(open(EXP / RUN / 'config.json'))
model = HTRNet(SimpleNamespace(**cfg['arch']), len(charset) + 1)
model.load_state_dict(
    torch.load(str(EXP / RUN / 'model.pt'), map_location=DEVICE, weights_only=False),
    strict=False
)
model = model.to(DEVICE).eval()
R = model.backbone.num_registers
print(f'Model : {RUN}  |  registers : {R}')

# ── Load & preprocess ─────────────────────────────────────────────────────────
raw    = load_image(str(ROOT / 'notebook' / 'sample_images' / IMG_FILE))
img    = preprocess(raw, (IMG_H, IMG_W))   # [128, 1024], inverted (0=white, high=ink)
tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
print(f'Image : {img.shape}  range [{img.min():.3f}, {img.max():.3f}]')

In [ ]:
# ── Cell 3 — CTC greedy decode ────────────────────────────────────────────────
with torch.no_grad():
    logits = model(tensor)   # [T, B, C]

probs = torch.softmax(logits[:, 0, :], dim=-1)   # [T, C]
ids   = probs.argmax(dim=-1).cpu().numpy()

chars_all, peaks_all = [], []
prev = -1
for t, idx in enumerate(ids):
    if idx != 0 and idx != prev:          # skip blank (0) and repeats
        ci = idx - 1
        if 0 <= ci < len(charset):
            chars_all.append(charset[ci])
            peaks_all.append(t)
    prev = idx

decoded = ''.join(chars_all)
print(f'Decoded : "{decoded}"')
print(f'All char–peak pairs : {list(zip(chars_all, peaks_all))}')

# Strip leading/trailing spaces for the figure (only show visible characters)
chars = [c for c in chars_all if c != ' ']
peaks = [p for c, p in zip(chars_all, peaks_all) if c != ' ']

print(f'\nVisible chars for grid : {list(zip(chars, peaks))}')

if len(chars) != TARGET_LEN:
    print(f'\n⚠  Expected {TARGET_LEN} visible chars but got {len(chars)}.  '
          f'Updating TARGET_LEN to {len(chars)}.')
    TARGET_LEN = len(chars)

print(f'\n✓  {len(chars)}-character word — uniform grid guaranteed.')

In [ ]:
# ── Cell 4 — Uniform character-aligned crop ───────────────────────────────────
#
# SCIENTIFIC RATIONALE:
# Naïve tight-crop to the ink bounding box makes the panel width depend on
# character width, so wide letters ("m", "w") get more pixels than narrow ones
# ("i", "l"). This biases the visual comparison.
#
# Instead we:
#  1. Tight-crop VERTICALLY (rows) — removes blank top/bottom margin.
#  2. Keep the FULL horizontal extent of the word for every panel.
#     Each panel therefore shows the same image region; only the saliency
#     overlay changes — a fair, apples-to-apples comparison.
#
# Optionally you may also compute per-character CTC column boundaries
# (character segmentation) and show those as thin vertical lines.

def vertical_crop(img, pad=8):
    """Crop blank rows only. img is inverted (0=white, high=ink)."""
    row_mean = img.mean(axis=1)          # [H]
    rt = row_mean.max() * 0.05
    rr = np.where(row_mean > rt)[0]
    if len(rr) == 0:
        return img, (0, img.shape[0])
    y0 = max(0, rr[0]  - pad)
    y1 = min(img.shape[0], rr[-1] + pad)
    return img[y0:y1, :], (y0, y1)

img_crop, (y0, y1) = vertical_crop(img)
print(f'Vertical crop → rows [{y0}:{y1}]  shape {img_crop.shape}')
print(f'Removed {100*(1 - img_crop.shape[0]/img.shape[0]):.0f}% blank vertical space')

# ── Optional: estimate character column boundaries from CTC peaks ─────────────
# Map each CTC time-step to a pixel column.
T  = logits.shape[0]
t_to_x = lambda t: int(round(t / T * IMG_W))   # linear mapping T → W

char_cols = [t_to_x(t) for t in peaks]          # left edge of each character
print(f'\nEstimated character x-positions: {list(zip(chars, char_cols))}')

fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ax.imshow(img_crop, cmap='gray_r', aspect='auto')
for x, ch in zip(char_cols, chars):
    ax.axvline(x, color='red', lw=0.8, alpha=0.6)
    ax.text(x + 3, 4, ch, color='red', fontsize=9, va='top')
ax.set_title(f'Vertically cropped input — "{decoded}"  (red lines = CTC peaks)', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 5 — Gradient saliency maps ──────────────────────────────────────────
#
# saliency(c, t) = |∂ logit[t, class(c)] / ∂ x|
#
# This is the most pixel-precise interpretability method available for a
# CTC model: it traces exactly how much each input pixel shifts the log-prob
# of character c at its CTC peak time-step t.

saliency_maps = []

for i, (ch, t) in enumerate(zip(chars, peaks)):
    inp = tensor.clone().detach().requires_grad_(True)
    model.zero_grad()
    lgts       = model(inp)                       # [T, B, C]
    class_idx  = charset.index(ch) + 1            # +1 because class 0 = CTC blank
    lgts[t, 0, class_idx].backward()

    grad      = inp.grad[0, 0].abs().cpu().numpy()   # [H, W]
    grad      = gaussian_filter(grad, sigma=2)         # slight spatial smoothing
    grad_crop = grad[y0:y1, :]                         # vertical crop only

    # Per-map min-max normalisation → [0, 1]
    gmin, gmax = grad_crop.min(), grad_crop.max()
    if gmax > gmin:
        grad_crop = (grad_crop - gmin) / (gmax - gmin)

    saliency_maps.append(grad_crop)
    print(f'  [{i}] "{ch}" @ t={t}, class={class_idx}  '
          f'grad range [{gmin:.4f}, {gmax:.4f}]')

print(f'\n✓  Computed {len(saliency_maps)} gradient saliency maps.')

In [ ]:
# ── Cell 6 — Self-attention maps ──────────────────────────────────────────────
#
# From the last transformer layer, for each character at CTC time-step t
# we extract the attention row of the corresponding sequence token
# (index R + t, since the first R slots are register tokens).
# We average over all attention heads for a clean summary.
#
# The resulting 1-D attention vector is upsampled to match img_crop width.

with torch.no_grad():
    seq_out, reg_out, attn_maps, norms, (Hp, Wp) = model.forward_explain(tensor)

# Last layer, mean over heads: [S, S]
attn_last = attn_maps[-1][0].mean(dim=0).numpy()
print(f'Attention matrix : {attn_last.shape}  |  R={R}  Wp={Wp}')

attn_char_maps = []

for i, (ch, t) in enumerate(zip(chars, peaks)):
    qi       = R + t                          # query = register offset + time
    row      = attn_last[qi, R:]              # attention to patch tokens [Wp]

    # Upsample 1-D patch attention to full image width
    scale    = IMG_W / len(row)
    row_up   = spzoom(row, scale, order=1)[:IMG_W]   # bilinear interp

    # Broadcast to 2-D (height-uniform — patches are 1-D for an HTR model)
    a_2d     = np.tile(row_up, (img_crop.shape[0], 1))   # [H_crop, W]

    # Normalise
    amin, amax = a_2d.min(), a_2d.max()
    if amax > amin:
        a_2d = (a_2d - amin) / (amax - amin)

    attn_char_maps.append(a_2d)
    print(f'  [{i}] "{ch}" @ t={t}, qi={qi}  attn range [{amin:.4f}, {amax:.4f}]')

print(f'\n✓  Computed {len(attn_char_maps)} self-attention maps.')

In [ ]:
# ── Cell 7 — Combined Fig. 5 reproduction ────────────────────────────────────
#
# Layout (matches paper Fig. 5):
#
#   Row 0 (Gradient Saliency):  [Input] | [c₁] | [c₂] | [c₃] | [c₄] | [c₅]
#   Row 1 (Self-Attention):     [Input] | [c₁] | [c₂] | [c₃] | [c₄] | [c₅]
#
# UNIFORM CROPPING:
#   Every character panel shows the SAME horizontal extent (full word width).
#   Only the heatmap overlay differs — the image substrate is identical.
#   This allows direct spatial comparison between characters without any
#   width-normalisation artefact.

n     = len(chars)
ncols = n + 1                   # input + N characters
nrows = 2                       # gradient saliency | self-attention

# ── Figure geometry ───────────────────────────────────────────────────────────
H_px, W_px = img_crop.shape
aspect      = H_px / W_px       # image aspect ratio

CELL_W = 2.4                    # inches per column
CELL_H = max(CELL_W * aspect, 0.9)

fig_w = CELL_W * ncols + 0.6
fig_h = CELL_H * nrows + 1.4

fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')

gs = gridspec.GridSpec(
    nrows, ncols,
    figure=fig,
    wspace=0.04, hspace=0.18,
    left=0.04, right=0.97,
    top=0.88, bottom=0.10
)

# ── Row labels ────────────────────────────────────────────────────────────────
ROW_LABELS = ['Gradient\nSaliency', 'Self-\nAttention']
CMAP       = 'inferno'

row_data = [
    (saliency_maps,   'Gradient Saliency'),
    (attn_char_maps,  'Self-Attention'),
]

for row, (maps, label) in enumerate(row_data):

    # ── Input column ──────────────────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[row, 0])
    ax0.imshow(img_crop, cmap='gray_r', aspect='auto', interpolation='lanczos')
    ax0.axis('off')

    # Row label on input panel
    ax0.set_ylabel(label, fontsize=10, fontweight='bold',
                   rotation=90, labelpad=6, va='center')
    ax0.yaxis.set_label_position('left')
    ax0.yaxis.label.set_visible(True)

    if row == 0:
        ax0.set_title('Input', fontsize=11, fontweight='bold', pad=5)

    # ── Character panels ──────────────────────────────────────────────────────
    for ci, (ch, hmap) in enumerate(zip(chars, maps)):
        ax = fig.add_subplot(gs[row, ci + 1])

        # Faded background — same image for every panel (UNIFORM)
        ax.imshow(img_crop, cmap='gray_r', aspect='auto',
                  interpolation='lanczos', alpha=0.25)

        # Saliency / attention overlay
        im = ax.imshow(hmap, cmap=CMAP, aspect='auto',
                       interpolation='lanczos', alpha=0.80,
                       vmin=0, vmax=1)
        ax.axis('off')

        if row == 0:
            ax.set_title(f"'{ch}'", fontsize=13, fontweight='bold', pad=5)

# ── Shared colour-bar ─────────────────────────────────────────────────────────
cbar_ax = fig.add_axes([0.97, 0.12, 0.015, 0.74])   # [left, bottom, w, h]
sm = plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, cax=cbar_ax)
cb.set_label('Normalised activation', fontsize=9)
cb.set_ticks([0, 0.5, 1])
cb.ax.tick_params(labelsize=8)

# ── Title ─────────────────────────────────────────────────────────────────────
fig.suptitle(
    f'Character-Level Attention Maps — "{decoded}"\n'
    f'ViT-RGTS  ({R} register tokens)  ·  {IMG_FILE}  ·  Uniform horizontal crop',
    fontsize=13, fontweight='bold', y=0.98
)

# ── Save ──────────────────────────────────────────────────────────────────────
out_png = str(OUT / 'fig5_combined_uniform.png')
out_pdf = str(OUT / 'fig5_combined_uniform.pdf')
fig.savefig(out_png, dpi=200, bbox_inches='tight', facecolor='white')
fig.savefig(out_pdf, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n✓  Saved:\n   {out_png}\n   {out_pdf}')

In [ ]:
# ── Cell 8 — Saliency-only panel (paper Fig. 5 exact layout) ─────────────────
#
# The reference figure shows ONE row: [Input | c₁ | c₂ | … | cₙ]
# This cell reproduces that minimalist layout for the gradient saliency only,
# which is the scientifically preferred method for a CTC model.

CELL_W2 = 2.2
CELL_H2 = max(CELL_W2 * aspect, 0.9)

fig2, axes2 = plt.subplots(
    1, ncols,
    figsize=(CELL_W2 * ncols + 0.4, CELL_H2 + 0.9),
    facecolor='white'
)

# Input
axes2[0].imshow(img_crop, cmap='gray_r', aspect='auto', interpolation='lanczos')
axes2[0].set_title('Input', fontsize=11, fontweight='bold', pad=5)
axes2[0].axis('off')

# Characters
for ci, (ch, sal) in enumerate(zip(chars, saliency_maps)):
    ax = axes2[ci + 1]
    ax.imshow(img_crop, cmap='gray_r', aspect='auto',
              interpolation='lanczos', alpha=0.25)
    ax.imshow(sal, cmap='inferno', aspect='auto',
              interpolation='lanczos', alpha=0.80, vmin=0, vmax=1)
    ax.set_title(f"'{ch}'", fontsize=13, fontweight='bold', pad=5)
    ax.axis('off')

fig2.suptitle(
    f'Character-Level Gradient Saliency — "{decoded}"\n'
    f'ViT-RGTS  ({R} registers)  ·  {IMG_FILE}  ·  Uniform crop',
    fontsize=12, fontweight='bold', y=1.04
)
plt.subplots_adjust(wspace=0.04)

out2 = str(OUT / 'fig5_saliency_uniform.png')
fig2.savefig(out2, dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f'✓  Saved: {out2}')

## Scientific notes

### Why uniform (horizontal) cropping matters

A naïve tight crop to each character's ink bounding box introduces a **width bias**: wider characters (m, w) occupy more pixels than narrow ones (i, l, t). When saliency maps are then overlaid on those differently-sized patches, the visual impression changes with character width rather than with the model's actual focus.  
By keeping **the same full-word region** in every panel and only varying the saliency overlay, we guarantee that spatial differences between panels are attributable solely to the attention/gradient signal.

### Why gradient saliency is preferred over raw attention weights

| Method | Pros | Cons |
|--------|------|------|
| **Gradient saliency** `|∂logit/∂x|` | Causal (measures actual pixel influence), pixel-level resolution, no architectural assumption | Requires backward pass per character |
| **Self-attention weights** | No backward pass needed, shows token-level routing | Attention ≠ attribution (Jain & Wallace 2019), 1-D (patch-level) only, can be misleading for multi-head models |

The combined figure (Cell 7) lets the reader judge both methods side-by-side.

### 5-character word choice

With exactly **5 characters** the figure panel count is fixed and the grid is symmetric. This avoids the irregular right-padding visible in the original paper figure (which uses a 6-character word including the period `.`).